# `EukaryoticToeholdGate` — usage example (trailing-Kozak layout)

A real, end-to-end run of the single-input eukaryotic toehold switch —
`EukaryoticToeholdGate` in `engine.gates.toehold`. This class is fully implemented
and tested (`tests/engine/gates/test_toehold.py`, 45 passing tests as of this
writing). This is a sibling to [`toehold.ipynb`](toehold.ipynb) (which drives the
gate through a stub `FoldEngine` for fast, dependency-free iteration and covers both
hosts generically) — here we build `EukaryoticToeholdGate` specifically and use the
**real** `FoldEngine` (ViennaRNA) throughout, so every number below is a genuine
fold, not a placeholder.

`ToeholdGate` builds two structurally different eukaryotic layouts (commits
`d754812`, `ee2f5f6`):

* **`"loop"`** — Kozak embedded in the hairpin loop, ported unmodified from the
  prokaryotic mechanism (steric occlusion of the start codon). Unvalidated for a
  eukaryotic toehold specifically — see `KOZAK_LAYOUTS`'s docstring. It also turns
  out Kozak and AUG are *not* adjacent in this layout (a 6 nt stem-closing segment
  sits between them), which breaks the Kozak consensus's own adjacency requirement.
* **`"trailing"`** — Kozak and the start codon sit *after* the closed hairpin
  instead, modelling scanning-ribosome blockage (docs/modalities.md) rather than
  direct start-codon occlusion. This is the layout the team's own eukaryotic
  scripts actually build (`plasmid_prefix + trg_bind_region + loop + stem_down +
  kozak` — Kozak last), and Kozak is genuinely adjacent to AUG here. It also carries
  no trailing `LINKER_SEQUENCE` — cap-dependent scanning initiates the instant the
  40S subunit meets Kozak+AUG, so nothing after the start codon matters to finding
  it, and the payload attaches directly.

It also optionally takes the real effector gene (`payload`, commit `74bf7b4`) and
folds its own first nucleotides into every design instead of a placeholder — because
the real downstream sequence can change which design actually scores best, not just
which one looks best in isolation.

This notebook builds the gate restricted to **`"trailing"`** only
(`kozak_layouts=("trailing",)`), since that's the layout that matches our own
established construction and is the newer, less-exercised code path, and pools
designs across the top several trigger candidates rather than just one.

No Django, no worker, no pipeline — just the gate class, constructed and called
directly, the way `pipeline.py` would use it internally.

## Setup

In [ ]:
# Put <repo>/src on the path. Search upward from cwd for pyproject.toml so this works
# wherever Jupyter is launched from.
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    if (_base / "pyproject.toml").exists():
        _src = _base / "src"
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from engine.domain import Host, Regulation, SelectedGene, TriggerSet, Constraints
from engine.gates.toehold import EukaryoticToeholdGate
from engine.gates.tools.folding import FoldEngine
from engine.gates.tools.translation import TranslationScorer
from engine.gates.tools.codons import CodonOptimizer
from engine.stages.folding import FoldProfiler
from engine.stages.motifs import MotifScreener
from engine.stages.off_target import OffTargetScanner
from engine.stages.triggers import TriggerScorer
from engine import sequences as sq

## 1. Build the tools, once

Per `CLAUDE.md` §5: tools are constructed once and handed to the gate, never built
inside a stage or family. `FoldEngine`'s cache is only useful if every caller shares
one instance — a second `FoldEngine()` means a cold cache and, worse, a second
chance to fold at a different temperature.

In [ ]:
host = Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track

folder = FoldEngine(temperature=37.0)
translation = TranslationScorer(host)
codons = CodonOptimizer(host)

# The real effector gene — from `eff` in CERNAL_FUNCTIONS.py (a GFP-family CDS, already
# RNA, 759 nt, tandem stop UAA UAA). When set, the gate folds its real first
# PAYLOAD_HEAD_LENGTH nt into every design instead of a placeholder — see
# PAYLOAD_HEAD_LENGTH's docstring (commit 74bf7b4) for why this can change which
# design scores best. Set to None to see the placeholder behaviour instead.
PAYLOAD_CDS = (
    "AUGCGUAAAGGAGAAGAACUUUUCACUGGAGUUGUCCCAAUUCUUGUUGAAUUAGAUGGUGAUGUUAAUGGGCACAAAUUUUCUGUCAG"
    "UGGAGAGGGUGAAGGUGAUGCAACAUACGGAAAACUUACCCUUAAAUUUAUUUGCACUACUGGAAAACUACCUGUUCCGUGGCCAACAC"
    "UUGUCACUACUUUCGGUUAUGGUGUUCAAUGCUUUGCGAGAUACCCAGAUCACAUGAAACAGCAUGACUUUUUCAAGAGUGCCAUGCCC"
    "GAAGGUUACGUACAGGAAAGAACUAUAUUUUUCAAAGAUGACGGGAACUACAAGACACGUGCUGAAGUCAAGUUUGAAGGUGAUACCCU"
    "UGUUAAUAGAAUCGAGUUAAAAGGUAUUGAUUUUAAAGAAGAUGGAAACAUUCUUGGACACAAAUUGGAAUACAACUAUAACUCACACA"
    "AUGUAUACAUCAUGGCAGACAAACAAAAGAAUGGAAUCAAAGUUAACUUCAAAAUUAGACACAACAUUGAAGAUGGAAGCGUUCAACUA"
    "GCAGACCAUUAUCAACAAAAUACUCCGAUUGGCGAUGGCCCUGUCCUUUUACCAGACAACCAUUACCUGUCCACACAAUCUGCCCUUUC"
    "GAAAGAUCCCAACGAAAAGAGAGACCACAUGGUCCUUCUUGAGUUUGUAACCGCUGCUGGGAUUACACAUGGCAUGGAUGAACUAUACA"
    "AAAGGCCUGCAGCAAACGACGAAAACUACGCUGCAUCAGUUUAAUAA"
)

# kozak_layouts restricts generate_designs to the "trailing" layout only — the default
# (omit this argument) sweeps both "loop" and "trailing" and lets engine.scoring rank
# across them. This mirrors how `host` is a constructor parameter rather than a
# subclass (docs/engine.md §2.4).
gate = EukaryoticToeholdGate(
    host, folder, translation, codons, kozak_layouts=("trailing",), payload=PAYLOAD_CDS
)
print(gate.required_tools())
print("kozak_layouts:", gate.kozak_layouts)
print("payload_head:", gate.payload_head)

## 2. Pick trigger candidates — via the real `TriggerScorer` (stage 2)

`TriggerScorer.score` is fully implemented (`tests/engine/test_triggers.py`, 47
passing tests): it slides every window of every length in
`constraints.trigger_lengths` across the transcript, screens out forbidden motifs,
folds each survivor for `openness`/`accessibility`/`mfe` via the same `FoldEngine`
the gate uses, checks off-targets, ranks by `accessibility * segment_specificity`,
and yields the top candidates per gene — so we use it for real here instead of
hand-picking one window.

`OffTargetScanner` itself is still a stub (`find_similar`/`scan_trigger` raise
`NotImplementedError`) — **except** when handed an empty transcriptome, which is a
deliberate early-return for exactly this case (a `direct` submission, or a demo like
this one, with no reference index to scan against): `scan_trigger` returns a clean
`OffTargetReport(hits=(), penalty=0.0)` rather than raising. That is a real,
documented behaviour of the class, not a workaround.

Earlier versions of this notebook carried only `candidates[0]` all the way through —
which meant only ever seeing one trigger's worth of designs. A real run explores every
surviving trigger, not just the single best-scoring one (a lower-ranked trigger can
still produce a better *switch* once folded), so this version carries the top
`N_TRIGGERS` through instead.

In [ ]:
# The full-length human AREG mRNA, as cDNA/DNA notation (T, not U) — same alphabet trap
# CLAUDE.md warns about, so normalise with to_rna() before anything else touches it.
transcript_dna = (
    "AGACGTTCGCACACCTGGGTGCCAGCGCCCCAGAGGTCCCGGGACAGCCCGAGGCGCCGCGCCCGCCGCCCCGAGCTCCCC"
    "AAGCCTTCGAGAGCGGCGCACACTCCCGGTCTCCACTCGCTCTTCCAACACCCGCTCGTTTTGGCGGCAGCTCGTGTCCCA"
    "GAGACCGAGTTGCCCCAGAGACCGAGACGCCGCCGCTGCGAAGGACCAATGAGAGCCCCGCTGCTACCGCCGGCGCCGGTG"
    "GTGCTGTCGCTCTTGATACTCGGCTCAGGCCATTATGCTGCTGGATTGGACCTCAATGACACCTACTCTGGGAAGCGTGAA"
    "CCATTTTCTGGGGACCACAGTGCTGATGGATTTGAGGTTACCTCAAGAAGTGAGATGTCTTCAGGGAGTGAGATTTCCCCT"
    "GTGAGTGAAATGCCTTCTAGTAGTGAACCGTCCTCGGGAGCCGACTATGACTACTCAGAAGAGTATGATAACGAACCACAA"
    "ATACCTGGCTATATTGTCGATGATTCAGTCAGAGTTGAACAGGTAGTTAAGCCCCCCCAAAACAAGACGGAAAGTGAAAAT"
    "ACTTCAGATAAACCCAAAAGAAAGAAAAAGGGAGGCAAAAATGGAAAAAATAGAAGAAACAGAAAGAAGAAAAATCCATGT"
    "AATGCAGAATTTCAAAATTTCTGCATTCACGGAGAATGCAAATATATAGAGCACCTGGAAGCAGTAACATGCAAATGTCA"
    "GCAAGAATATTTCGGTGAACGGTGTGGGGAAAAGTCCATGAAAACTCACAGCATGATTGACAGTAGTTTATCAAAAATTG"
    "CATTAGCAGCCATAGCTGCCTTTATGTCTGCTGTGATCCTCACAGCTGTTGCTGTTATTACAGTCCAGCTTAGAAGACAA"
    "TACGTCAGGAAATATGAAGGAGAAGCTGAGGAACGAAAGAAACTTCGACAAGAGAATGGAAATGTACATGCTATAGCATA"
    "ACTGAAGATAAAATTACAGGATATCACATTGGAGTCACTGCCAAGTCATAGCCATAAATGATGAGTCGGTCCTCTTTCCA"
    "GTGGATCATAAGACAATGGACCCTTTTTGTTATGATGGTTTTAAACTTTCAATTGTCACTTTTTATGCTATTTCTGTATA"
    "TAAAGGTGCACGAAGGTAAAAAGTATTTTTTCAAGTTGTAAATAATTTATTTAATATTTAATGGAAGTGTATTTATTTTA"
    "CAGCTCATTAAACTTTTTTAACCAAA"
)
transcript = sq.to_rna(transcript_dna)
assert sq.is_valid_rna(transcript)
print(f"transcript length: {len(transcript)} nt")

In [ ]:
# Stage-2 tools, built once (same injection rule as the gate's own tools).
profiler = FoldProfiler()
screener = MotifScreener()
off_target = OffTargetScanner(transcriptome={})  # empty: no reference index for this demo
scorer = TriggerScorer(profiler, off_target, screener, folder)  # shares the gate's FoldEngine

# In a real run this comes from GeneSelector (stage 1); stand in with a minimal
# SelectedGene since this demo starts from a single known transcript.
gene = SelectedGene(
    gene_id="AREG",
    symbol="AREG",
    regulation=Regulation.UP,
    log2_fold_change=2.0,
    score=1.0,
)
constraints = Constraints(trigger_lengths=(30, 36), max_switch_length=200)

candidates = list(scorer.score([gene], {"AREG": transcript}, constraints))
print(f"{len(candidates)} candidate(s), best-scoring first\n")
for c in candidates[:5]:
    print(
        f"  {c.trigger_id:22} start={c.start_index:4} len={c.length:2}  "
        f"score={c.score:.3f}  accessibility={c.accessibility:.3f}  gc={c.gc_content:.1f}"
    )

N_TRIGGERS = 5
top_triggers = candidates[:N_TRIGGERS]
print(f"\ncarrying the top {len(top_triggers)} trigger(s) forward")

## 3. Wrap each trigger in its own `TriggerSet`

`TriggerSet` is the circuit's inputs (one activator each here — every trigger builds
its own single-input switch, independently). `Constraints` were already built above,
since `TriggerScorer` needed them too — a run builds `Constraints` once from
`params["constraints"]` and threads the same object through every stage.

In [ ]:
trigger_sets = [TriggerSet(activators=(t,)) for t in top_triggers]

for ts in trigger_sets:
    print(ts.activators[0].trigger_id, "| arity:", ts.arity, "| logic:", ts.logic_type)

## 4. `is_compatible()` — cheap check before generating anything

Arity, host, trigger length window — nothing here folds. Checked per trigger set,
same as a real run would (a family is checked against every trigger set it might
build from).

In [ ]:
compatible_trigger_sets = []
for ts in trigger_sets:
    compatibility = gate.is_compatible(ts, constraints)
    print(ts.activators[0].trigger_id, compatibility)
    if compatibility.ok:
        compatible_trigger_sets.append(ts)

assert compatible_trigger_sets, "no trigger set survived is_compatible"

## 5. `generate_designs()` — the full candidate pool

Called **once per compatible trigger set**, not just the top one — exactly how a real
run explores the search space, since a lower-ranked trigger can still fold into a
better switch. With `kozak_layouts=("trailing",)`, each trigger set yields one
`GateDesign` per `toehold_lengths` x `TRAILING_LOOP_LENGTHS` x `KOZAK_LINKER_LENGTHS`
combination it supports. Pooled together, this is the candidate set `engine.scoring`
would actually rank in a real run.

In [ ]:
designs = []
for ts in compatible_trigger_sets:
    designs.extend(gate.generate_designs(ts, constraints))

print(f"{len(designs)} design(s) across {len(compatible_trigger_sets)} trigger(s)\n")
for d in designs:
    print(
        f"  {d.design_id:52} {d.length:3} nt  "
        f"trigger={d.trigger_set.activators[0].trigger_id:18}  "
        f"loop_len={d.architecture['loop_len']:2}  "
        f"kozak_linker_len={d.architecture['kozak_linker_len']}"
    )

## 6. `evaluate_design()` — raw metrics and sequences, across the whole pool

**Raw** values only — no normalising, weighting or ranking here, that is
`engine.scoring`'s job. Keys are exactly the metric names `DEFAULT_V1` declares.

For the `"trailing"` layout specifically, `predicted_leakage`/`dynamic_range` are read
from the **toehold+stem region**, not the AUG — the AUG sits outside the hairpin here
and stays roughly accessible whether or not the trigger is bound, so AUG-region
accessibility would not discriminate ON from OFF for this layout (see
`evaluate_design`'s docstring). This is a new, unreviewed proxy — not a port of
anything previously validated.

Because `PAYLOAD_CDS` is set (cell 4), every design below is folded **with the real
effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides**, not a placeholder — the
molecule `gate_folding_energy`/`predicted_leakage`/`dynamic_range` are measured on is
the one a ribosome would actually encounter once this gene is attached. Set
`PAYLOAD_CDS = None` in cell 4 and re-run to see how the numbers shift for the exact
same designs without that information.

The cell below prints the metrics table first, then every design's synthesis-ready
sequence (`emit_sequence()`) in the same best-first order, so you can see both
together rather than metrics alone.

In [ ]:
all_metrics = [(d, gate.evaluate_design(d)) for d in designs]
ranked = sorted(all_metrics, key=lambda pair: pair[1]["dynamic_range"], reverse=True)

print(
    f"{'trigger':>18}  {'loop_len':>8}  {'linker_len':>10}  {'leakage':>8}  "
    f"{'dyn_range':>9}  {'folding_energy':>14}"
)
for d, m in ranked:
    print(
        f"{d.trigger_set.activators[0].trigger_id:>18}  "
        f"{d.architecture['loop_len']:>8}  {d.architecture['kozak_linker_len']:>10}  "
        f"{m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{m['gate_folding_energy']:>14.1f}"
    )

print("\nsequences, same order (best first):\n")
for d, m in ranked:
    print(f"{d.design_id}  ({d.length} nt)")
    print(f"  {gate.emit_sequence(d)}")

# Not this gate's job to rank in a real run (engine.scoring owns that) — but picking the
# top-ranked one here to carry through the rest of this notebook's cells.
design, metrics = ranked[0]
print(f"\nbest by dynamic_range: {design.design_id}")
for name, value in metrics.items():
    print(f"  {name:22} {value}")

Two things worth noticing in that table:

1. **The best switch did not come from the best-scoring trigger.** `TriggerScorer`
   ranked `trig-AREG-111-30` highest by its own `score` (accessibility x
   segment_specificity — cell 7), but the best-*switch* by `dynamic_range` came from
   `trig-AREG-112-30`. A trigger that looks slightly weaker on its own can still fold
   into a better gate once the actual hairpin construction and the payload are
   accounted for — exactly why a real run pools **every** surviving trigger through
   `generate_designs` rather than stopping at the single top-ranked one. (Swap
   `PAYLOAD_CDS` for a different gene or `None` in cell 4 and re-run: the winner
   changes again — this ranking genuinely depends on which effector gene is attached,
   not just on the trigger or the switch geometry.)
2. **`dynamic_range` (higher is better) still comes out below 1.0 everywhere.** The
   toehold+stem region gets *less* accessible, not more, when the trigger binds, for
   every design in this pool. `predicted_leakage` is comfortably under the 0.85
   hard-filter threshold throughout, so nothing here would be rejected outright, but
   `engine.scoring`'s ranking would still push all of these toward the bottom relative
   to a design that actually opens on trigger binding. That's a real result about this
   trigger region and `toehold_length=12` specifically, not something to paper over —
   try widening `constraints.trigger_lengths`, or scanning a different part of the
   transcript, to see whether the mechanism performs better elsewhere.

## 7. `emit_sequence()` — the synthesis-ready sequence

In [ ]:
print(gate.emit_sequence(design))

The Kozak element and start codon aren't visually obvious in that raw string — for this
`"trailing"` layout they sit right after the closed hairpin, not inside it (contrast
with `"loop"`, where they'd be buried in the middle). Note also what comes right after
the AUG here: the real effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides
(`eff` from `CERNAL_FUNCTIONS.py`, set as `PAYLOAD_CDS` in cell 4), not a placeholder
— `"trailing"` carries no trailing linker of its own (commit `ee2f5f6`); cap-dependent
scanning initiates the instant the 40S subunit meets Kozak+AUG, so nothing after the
start codon plays any role in finding it, and whatever comes after is either the
payload (when known, as folded in here) or nothing at all (commit `74bf7b4`). Locate
Kozak and the start codon explicitly using `design.architecture` (which records
`aug_index` exactly) and the gate's own `KOZAK_EUKARYOTIC` constant:

In [ ]:
seq = design.sequence
aug_index = design.architecture["aug_index"]
kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
assert kozak_index != -1, "Kozak element not found — architecture assumptions above are stale"

marks = [" "] * len(seq)
for i in range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)):
    marks[i] = "K"
for i in range(aug_index, aug_index + 3):
    marks[i] = "A"

print(seq)
print("".join(marks), " K = Kozak (GCCACC)   A = start codon (AUG)")

## Where this fits in a real run

This notebook now pools 20 designs across 5 triggers and one layout — closer to a real
run, but still a small corner of it. In the actual pipeline:

- Every trigger `TriggerScorer` keeps (not just the top `N_TRIGGERS`) gets checked with
  `is_compatible` and, if it passes, run through `generate_designs` — and by default
  that sweeps **both** `"loop"` and `"trailing"` layouts (this notebook restricted to
  `"trailing"` only via `kozak_layouts` — see cell 4).
- Every design's raw metrics go through `engine.scoring` — `build_metrics`,
  `weighted_score`, `failed_filter`, `rank_candidates` — which is what actually
  decides which designs survive and how they rank, comparably with every other gate
  family's designs, every trigger, **and across layouts**. Neither this notebook nor
  the gate itself picks a winner — sorting by `dynamic_range` alone (cell 15) is a
  stand-in for that, not the real ranking.
- `CandidateStore` records provenance and writes the stage snapshot; nothing here
  hand-rolls a results CSV.
- The **payload** folds into evaluation when known (`PAYLOAD_CDS` in cell 4) but the
  *actual* fused construct is still assembled later, at plasmid assembly
  (`PlasmidBuilder.build(circuit, DesiredOutcome.CUSTOM, custom_payload=...)`), using
  the complete gene, not just its folded-in head.

The two-input AND version (`EukaryoticToeholdAndGate`) is **not** implemented yet —
its `generate_designs` still raises `NotImplementedError("Step 5")`. This notebook
only covers the single-input case.

Open questions this work surfaced, not resolved here (see commits `d754812`,
`ee2f5f6`, `74bf7b4`): whether `"loop"` should still ship as the eukaryotic default
now that `"trailing"` exists (and now that `"loop"`'s Kozak-AUG adjacency is known to
be broken); whether `TRAILING_LOOP_LENGTHS`/`KOZAK_LINKER_LENGTHS` are the right
ranges to sweep; whether the `"trailing"` leakage proxy (toehold+stem accessibility)
is the right measurement for scanning-ribosome blockage at all; and whether fusing the
payload directly after the switch's own placeholder AUG (rather than dropping that AUG
in favour of the payload's own) is the right call — the fully-assembled ORF still
reads switch-AUG then the payload's required leading AUG as an ordinary internal
codon, regardless of the payload head folded in here for evaluation.